# DATA Hypothesis - VANGUARD A/B TEST
---



**How effective was the redesign?**
Database Client Profiles (df_exde): Demographics like age, gender, and account details of our clients. The database is clean.
Web experiment (web): A detailed trace of client interactions online.

IMPORT LIBRARIES

In [42]:
import pandas as pd
import numpy as np
import scipy.stats as st

1. LOAD THE DATASETS 

In [43]:
data_path = "data/processed"
df_exde = pd.read_csv(f"{data_path}/df_exde.csv")
vanguard_client_summary = pd.read_csv(f"{data_path}/vanguard_client_summary.csv")

In [44]:
vanguard_hypo = pd.merge(df_exde,vanguard_client_summary,on=['client_id', 'variation'],how='outer')
vanguard_hypo

,client_id,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,gender,num_accts,balance,calls_6_mnth,logons_6_mnth,variation,...,visit_id,process_step,process_order,process_status,date_time,next_time,is_global_success,is_completed,has_backwards_error,seconds_spent
0,555,3.0,46.0,29.5,unknown,2.0,25454.66,2.0,6.0,Test,...,637149525_38041617439_716659,start,step_1,correct,2017-04-15 12:57:56,2017-04-15 12:58:03,1,1,0,7.0
1,555,3.0,46.0,29.5,unknown,2.0,25454.66,2.0,6.0,Test,...,637149525_38041617439_716659,step_1,step_2,correct,2017-04-15 12:58:03,2017-04-15 12:58:35,1,1,0,32.0
2,555,3.0,46.0,29.5,unknown,2.0,25454.66,2.0,6.0,Test,...,637149525_38041617439_716659,step_2,step_3,correct,2017-04-15 12:58:35,2017-04-15 13:00:14,1,1,0,99.0
3,555,3.0,46.0,29.5,unknown,2.0,25454.66,2.0,6.0,Test,...,637149525_38041617439_716659,step_3,confirm,correct,2017-04-15 13:00:14,2017-04-15 13:00:34,1,1,0,20.0
4,555,3.0,46.0,29.5,unknown,2.0,25454.66,2.0,6.0,Test,...,637149525_38041617439_716659,confirm,NaN,correct,2017-04-15 13:00:34,NaN,1,1,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
317230,9999729,10.0,124.0,31.0,female,3.0,107059.74,6.0,9.0,Test,...,870243567_56915814033_814203,confirm,NaN,correct,2017-05-08 16:09:40,NaN,0,1,0,NaN
317231,9999729,10.0,124.0,31.0,female,3.0,107059.74,6.0,9.0,Test,...,99583652_41711450505_426179,start,step_1,correct,2017-04-05 13:40:49,2017-04-05 13:41:04,0,1,0,15.0
317232,9999729,10.0,124.0,31.0,female,3.0,107059.74,6.0,9.0,Test,...,99583652_41711450505_426179,step_1,NaN,error,2017-04-05 13:41:04,NaN,0,1,1,NaN
317233,9999832,23.0,281.0,49.0,female,2.0,431887.61,1.0,4.0,Test,...,472154369_16714624241_585315,start,step_1,correct,2017-05-16 16:46:03,2017-05-16 16:46:11,0,0,0,8.0


In [45]:
vanguard_hypo.columns

Index(['client_id', 'clnt_tenure_yr', 'clnt_tenure_mnth', 'clnt_age', 'gender',
       'num_accts', 'balance', 'calls_6_mnth', 'logons_6_mnth', 'variation',
       'age_group', 'visitor_id', 'visit_id', 'process_step', 'process_order',
       'process_status', 'date_time', 'next_time', 'is_global_success',
       'is_completed', 'has_backwards_error', 'seconds_spent'],
      dtype='object')

## Phase 3 — Hypothesis Testing

## Hypothesis 1

**Completion Rate** Completation rate with repetations
H0: Completion rate is the same in the control group and test group.
H1: Completion rate is different in from the control group.

*Choose significance level*

In [46]:
alpha = 0.05

*Compute Test Statistic*

In [47]:
from statsmodels.stats.proportion import proportions_ztest

client_summary = (vanguard_hypo.groupby(['variation', 'client_id'])['is_completed']
    .max()  # True if client completed at least once
    .reset_index())

summary = client_summary.groupby('variation')['is_completed'].agg(['sum', 'count']).reset_index()
print(summary)

count = summary['sum'].values      # number of clients who completed
nobs = summary['count'].values     # total number of unique clients

stat, p = proportions_ztest(count, nobs)
print("z-stat:", stat)
print("p-value:", p)

  variation    sum  count
0   Control  15434  23532
1      Test  18687  26968
z-stat: -8.8745141890702
p-value: 7.023933247581432e-19


*Determine p_value*

In [48]:
p_value = 7.023933247581432e-19

*Decision Making*

In [49]:
if p_value > alpha:
    print("Null hypothesis accepted, H0: Completion rate is the same in the control group and test group")
else:
    print("Rejected the null hypothesis, H1 accepted: Completion rate is different in from the control group.")

Rejected the null hypothesis, H1 accepted: Completion rate is different in from the control group.


**Result Interpretation**: There is significative difference between the test group and the control group (p<0.05). Because the z-stat is minor -8.8745. The Test group has a higher completation rate that the control group. 

## Hypothesis 2 with is global_sucess: 
Global completion rate without repetations and errors. 

**Global Completion Rate** Completation rate without repetations
H0: Global Completion rate is the same in the control group and test group.
H1: Global Completion rate is different in from the control group.

In [50]:
alpha = 0.05

In [51]:
from statsmodels.stats.proportion import proportions_ztest

client_summary = (vanguard_hypo.groupby(['variation', 'client_id'])['is_global_success']
    .max()  # True if client completed at least once
    .reset_index())

summary = client_summary.groupby('variation')['is_global_success'].agg(['sum', 'count']).reset_index()
print(summary)

count = summary['sum'].values      # number of clients who completed
nobs = summary['count'].values     # total number of unique clients

stat, p = proportions_ztest(count, nobs)
print("z-stat:", stat)
print("p-value:", p)


  variation   sum  count
0   Control  6916  23532
1      Test  8014  26968
z-stat: -0.8031338477959028
p-value: 0.42189737581314724


In [52]:
p_value = 0.42189737581314724

In [53]:
if p_value > alpha:
    print("Null hypothesis accepted, H0: Global Completion rate is the same in the control group and test group.")
else:
    print("Rejected the null hypothesis, H1 accepted: Global Completion rate is different in from the control group.")

Null hypothesis accepted, H0: Global Completion rate is the same in the control group and test group.


**Result**: The test show that the Global completation rate is the same in both groups (p < 0.05)

## Hypothesis 3

**Completion Rate with a Cost-Effectiveness Threshold**
H0: The Test group does not improve completion rate over Control by at least 5%.
H1: The Test group improves completion rate over Control by more than 5%. 

*Choose significance level*

In [54]:
alpha = 0.05

*Compute Test Statistic*

In [55]:
from scipy.stats import norm

# Data
sum_control = 15434
count_control = 23532

sum_test = 18687
count_test = 26968

# Proportions
p_control = sum_control / count_control
p_test = sum_test / count_test

# Pooled proportion (FIXED)
p_pool = (sum_control + sum_test) / (count_control + count_test)

# Standard error
se = np.sqrt(p_pool * (1 - p_pool) * (1/count_control + 1/count_test))

# Threshold (5% absolute uplift)
delta = 0.05

# Z statistic for uplift test
z = ((p_test - p_control) - delta) / se

# One-sided p-value
p_value = 1 - norm.cdf(z)

# Output
print("Control rate:", p_control)
print("Test rate:", p_test)
print("Observed uplift:", p_test - p_control)
print("Z-stat:", z)
print("P-value:", p_value)

Control rate: 0.6558728539860615
Test rate: 0.6929323642835954
Observed uplift: 0.03705951029753385
Z-stat: -3.0988148131491866
P-value: 0.999028517875349


*Determine p_value*

In [56]:
p_value = 0.999028517875349

In [57]:
if p_value > alpha:
    print("Null hypothesis accepted, H0: The Test group does not improve completion rate over Control by at least 5%.")
else:
    print("Rejected the null hypothesis, The Test group improves completion rate over Control by more than 5%.")

Null hypothesis accepted, H0: The Test group does not improve completion rate over Control by at least 5%.


**Result Interpretation**: The z-test shows an improvement of completion rate of the test group of 3.7% over the control group. However this increment does not show a significant difference between groups(p > 0.05). 

## Hypothesis 4

**Error Rate according to age**
H₀: The error rate is the same in control and test groups
H₁: The error rate differs between control and test groups

*Choose significance level*

In [58]:
alpha = 0.05

*Data Collection*

In [59]:
from statsmodels.stats.proportion import proportions_ztest

error_summary = (vanguard_hypo.groupby(['variation', 'client_id'])['has_backwards_error']
    .max()  # True if client completed at least once
    .reset_index())

error = error_summary.groupby('variation')['has_backwards_error'].agg(['sum', 'count']).reset_index()
print(error)

count = summary['sum'].values      # number of clients who completed
nobs = summary['count'].values     # total number of unique clients

stat, p = proportions_ztest(count, nobs)
print("z-stat:", stat)
print("p-value:", p)


  variation    sum  count
0   Control  14418  23532
1      Test  15251  26968
z-stat: -0.8031338477959028
p-value: 0.42189737581314724


In [60]:
p_value = 0.42189737581314724

*Compute Test Statistic*

In [61]:
if p_value > alpha:
    print("Null hypothesis accepted, H0: The error rate is the same in control and test groups")
else:
    print("Rejected the null hypothesis, then H1 accepted: The error rate differs between control and test groups")

Null hypothesis accepted, H0: The error rate is the same in control and test groups


**Result Interpretation**: Even though the total count of errors in the test control is higher showing a negative z-value. The z-test shows the error rate does not have 
a significative difference in both groups (p-value > 0.05). 

## Hypothesis 5

H0: The mean of time spend on the User Interface (UI) is the same in Control and Test group
H1: The mean of time spend on the User Interface (UI) is the different in control and test group

*Choose significance level*

In [62]:
alpha = 0.05

*Data Collection*

In [63]:
time_test = vanguard_hypo[
    vanguard_hypo['variation'] == "Test"]['seconds_spent'].mean()

print(time_test)

84.13366070468476


In [64]:
time_control = vanguard_hypo[
    vanguard_hypo['variation'] == "Control"]["seconds_spent"].mean()
print(time_control)

83.51883316557857


*Compute Test Statistic*

In [65]:
from scipy import stats
import pandas as pd

# 1. Extract the WHOLE columns (distributions), not just a single mean
# We use the raw data columns from your dataframe
raw_test = vanguard_hypo[vanguard_hypo['variation'] == 'Test']['seconds_spent']
raw_control = vanguard_hypo[vanguard_hypo['variation'] == 'Control']['seconds_spent']

# 2. Clean them using pd.to_numeric and dropna
# This works now because 'raw_test' is a Series, not a single float
time_test = pd.to_numeric(raw_test, errors='coerce').dropna()
time_control = pd.to_numeric(raw_control, errors='coerce').dropna()

# 3. Run the T-test
t_stat, p_value = stats.ttest_ind(time_test, time_control, equal_var=False)

print("t-stat:", t_stat)
print("p-value:", p_value)

t-stat: 0.697587842041964
p-value: 0.4854357371693778


In [66]:
p_value = 0.4854357371693778
alpha = 0.05

In [67]:
if p_value > alpha:
    print("Null hypothesis accepted, H0: The mean of time spend on the User Interface (UI) is the same in Control and Test group")
else:
    print("Rejected the null hypothesis, then H1 accepted: The mean of time spend on the User Interface (UI) is the different in control and test group")

Null hypothesis accepted, H0: The mean of time spend on the User Interface (UI) is the same in Control and Test group


**Interpretation results**: The average time spend on the User Interface (UI) is not significatly different (p > 0.05) between groups.

Mann-Whitney test to double check the last hypothesis

In [68]:
from scipy.stats import mannwhitneyu

# Run the test on the cleaned distributions from your previous step
u_stat, p_value = mannwhitneyu(time_test, time_control, alternative='two-sided')

print(f"Mann-Whitney U Statistic: {u_stat}")
print(f"P-value: {p_value}")

# Interpretation
if p_value > 0.05:
    print("Null hypothesis accepted, H0: The mean of time spend on the User Interface (UI) is the same in Control and Test group")
else:
    print("Rejected the null hypothesis, then H1 accepted: The mean of time spend on the User Interface (UI) is the different in control and test group")

Mann-Whitney U Statistic: 7429638575.0
P-value: 2.6584706694915636e-12
Rejected the null hypothesis, then H1 accepted: The mean of time spend on the User Interface (UI) is the different in control and test group


In [69]:
import os

# Define the output directory
output_dir = "data/processed"
os.makedirs(output_dir, exist_ok=True)

vanguard_hypo.to_csv(f"{output_dir}/vanguard_hypo.csv", index=False)

print("Exported:")
for f in os.listdir(output_dir):
    path = os.path.join(output_dir, f)
    if os.path.isfile(path):
        size = os.path.getsize(path)
        print(f"  {f} ({size:,} bytes)")

Exported:
  df_exde.csv (3,302,143 bytes)
  df_web_3.csv (81,420,206 bytes)
  vanguard_client_summary.csv (41,859,699 bytes)
  vanguard_hypo.csv (57,778,112 bytes)
  web.csv (39,957,052 bytes)
